In [0]:
import requests
import json

def scrape_reliefweb_articles(app_name, limit=2):
    """
    Scrapes articles from the ReliefWeb API main headlines page, extracting Title, Body, and Date.
    Only uses data from the list response - no individual article detail calls.
    """
    base_url = "https://api.reliefweb.int/v2/reports"
    print(f"Fetching the top {limit} articles from headlines...")
    
    # ReliefWeb API requires POST requests for filtered queries
    # appname must be a query parameter
    # view=headlines fetches articles from the main headlines/featured section
    query_params = {
        "appname": app_name
    }
    
    # Filter for English language only
    # Note: NOT operator is not supported by the API, so we filter out maps/infographics client-side
    payload = {
  "filter": {
    "field": "headline"
  },
  "offset": 0,
  "limit": 20,
  "preset": "latest",
  "profile": "list"
}
    
    try:
        response = requests.post(base_url, params=query_params, json=payload)
        response.raise_for_status()
        print(f"URL: {response.url}")
        print(f"Status code: {response.status_code}")
        list_data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching article list: {e}")
        return []

    articles = list_data.get("data", [])
    extracted_reports = []

    for item in articles:
        article_id = item.get("id")
        if not article_id:
            continue
        
        # Get fields from the list response
        fields = item.get("fields", {})
        
        # Filter out maps and infographics client-side
        format_list = fields.get("format", [])
        format_names = [f.get("name", "") for f in format_list]
        if "Map" in format_names or "Infographic" in format_names:
            print(f"Skipping article ID {article_id} (Map/Infographic)")
            continue
        
        # Extract data directly from the list response
        title = fields.get("title", "No Title Available")
        body = fields.get("body", "No Body Available")
        date_info = fields.get("date", {})
        published_date = date_info.get("original", "No Date Available")
        
        extracted_reports.append({
            "id": article_id,
            "title": title,
            "published_date": published_date,
            "body": body
        })
        
        print(f"Extracted article ID: {article_id}")

    print(f"\nTotal articles extracted: {len(extracted_reports)}")
    return extracted_reports


In [0]:
# --- DATABRICKS EXECUTION AND SAVING ---

# 1. Run the scraper
APP_NAME = "datathon-quattroformaggi-hum-7x4k-2025" 
results = scrape_reliefweb_articles(APP_NAME, limit=10)  # Reduced from 1000 to 100 to avoid API 500 errors

if results:
    # 2. Convert the Python list of dictionaries into a PySpark DataFrame
    # Note: 'spark' is a built-in session variable in Databricks notebooks
    df = spark.createDataFrame(results)
    
    # 3. Display the dataframe in the notebook UI
    display(df)
    
    # 4. Save the dataframe as a SQL Table (Delta format)
    # Change "default.reliefweb_articles" to your preferred database/schema and table name
    table_name = "unocha.default.reliefweb_articles_selected"
    
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(table_name)
      
    print(f"\nSuccess! Data saved to SQL table: {table_name}")
else:
    print("\nNo data was extracted, so no table was created.")

In [0]:
import requests
import json

def scrape_reliefweb_articles(app_name, limit=20):
    """
    Pobiera artykuły z API ReliefWeb używając metody POST i zapytania z payloadem,
    oraz filtruje mapy/infografiki po stronie klienta.
    """
    base_url = "https://api.reliefweb.int/v2/reports"
    print(f"Fetching the top {limit} articles using POST payload...")
    
    # Parametry przekazywane w URL
    query_params = {
        "appname": app_name
    }
    
    # Ciało zapytania (body) przekazywane jako JSON
    payload = {
        "filter": {
            "field": "headline"
        },
        "offset": 0,
        "limit": limit, # Używamy zmiennej z funkcji
        "preset": "latest",
        "profile": "list"
    }
    
    try:
        # Zmiana: Używamy POST i przekazujemy json=payload
        response = requests.post(base_url, params=query_params, json=payload)
        response.raise_for_status()
        list_data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching article list: {e}")
        return []

    articles = list_data.get("data", [])
    extracted_reports = []

    for item in articles:
        article_id = item.get("id")
        if not article_id:
            continue
            
        print(f"Fetching details for article ID: {article_id}...")
        detail_url = f"{base_url}/{article_id}"
        detail_params = {"appname": app_name}
        
        try:
            detail_response = requests.get(detail_url, params=detail_params)
            detail_response.raise_for_status()
            detail_data = detail_response.json()
            
            if detail_data.get("data") and len(detail_data["data"]) > 0:
                fields = detail_data["data"][0].get("fields", {})
                
                # --- FILTROWANIE PO STRONIE KLIENTA (Client-side filtering) ---
                # Sprawdzamy pole format i odrzucamy, jeśli to Mapa lub Infografika
                formats = fields.get("format", [])
                format_names = [f.get("name") for f in formats]
                if "Map" in format_names or "Infographic" in format_names:
                    print(f" -> Pomijam ID {article_id} (Format: {format_names})")
                    continue
                # --------------------------------------------------------------
                
                title = fields.get("title", "No Title Available")
                body = fields.get("body", "No Body Available")
                date_info = fields.get("date", {})
                published_date = date_info.get("original", "No Date Available")
                
                extracted_reports.append({
                    "id": article_id,
                    "title": title,
                    "published_date": published_date,
                    "body": body
                })
                
        except requests.exceptions.RequestException as e:
            print(f"Error fetching details for ID {article_id}: {e}")

    return extracted_reports

# --- DATABRICKS EXECUTION ---

APP_NAME = "datathon-quattroformaggi-hum-7x4k-2025" 
# Limit ustawiony na 20, zgodnie z Twoim payloadem
results = scrape_reliefweb_articles(APP_NAME, limit=1000)

if results:
    df = spark.createDataFrame(results)
    display(df)
    table_name = "unocha.default.reliefweb_articles_selected"
    
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(table_name)
    
    # Opcjonalny zapis:
    # df.write.format("delta").mode("overwrite").saveAsTable("default.reliefweb_articles_newest")